# Music Annotation Project
## Notebook 72: Frozen V8.8 Annotated-Only Supervised Operational Trial

**Runtime:** CPU by default. Use GPU only if the resolved frozen V8.8 adapter requires it.

This notebook processes every eligible reviewed page from `source/annotated_only` and `source/annotated` through one identical clean-free runtime path. It locks inputs, decisions, probabilities and corrected pages before resolving `source/clean`. Clean truth is used only after the prediction lock for paired-page scoring.

### Fixed contract
- reviewer-corrected authoritative mask is the sole intervention authority;
- no `source/clean` access before the prediction lock is CLOSED;
- frozen V8.8 only: no training, refitting, recalibration, threshold movement or policy change;
- zero changed pixels outside the prepared authoritative mask;
- exact frozen fallback behaviour outside valid V8.8 interventions;
- every candidate page is processed, excluded or failed with an explicit reason;
- every processed page receives a blinded original-versus-corrected usability package;
- no automatic production or mask-authority release.

In [20]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow gradio pillow scikit-image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Configuration and immutable run directory

For the definitive run, pin `V88_OPERATOR_PATH` and `V88_BUNDLE_ROOT` to the exact frozen release. The preferred adapter is a durable `v8_8_operator.py` exposing `load_operator(...)` and `run_page(observed, authoritative_seed_mask, runtime_context)`.

In [21]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
import builtins, contextlib, hashlib, importlib.util, json, os, platform, random, shutil, sys, time, traceback, zipfile
import cv2, numpy as np, pandas as pd

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
ANNOTATED_ONLY_DIR=ROOT/'source'/'annotated_only'
ANNOTATED_DIR=ROOT/'source'/'annotated'
# Deliberately a string before lock. Do not instantiate Path or enumerate it pre-lock.
CLEAN_DIR_LITERAL='/content/drive/MyDrive/MusicAnnotationProject/source/clean'
HITL_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
HITL_DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEWED_MASK_FALLBACK_DIRS=[ROOT/'production_pilot'/'reviewed_masks', ROOT/'human_in_the_middle'/'corrected_masks']
OUTPUT_ROOT=ROOT/'notebook72_runs'

# Exact frozen identities. Set overrides for the definitive run.
V88_OPERATOR_PATH=ROOT/'v8_8_release'/'v8_8_operator.py'
V88_BUNDLE_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'/'20261106_210922_v8_8b_notebook53R3'
ACTIVE_RELEASE_POINTER=ROOT/'v8_8_registered_replay_release_control'/'active_release.json'

SMOKE_TEST=False
SMOKE_TEST_PAGES=5
PAGE_ALLOWLIST=None
PUBLIC_LINK=True
RANDOM_SEED=72
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
EXPECTED_API_VERSION='notebook72-v1'

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72_frozen_v8_8_operational_trial'
OUT=OUTPUT_ROOT/RUN_ID
D00=OUT/'00_configuration'; D01=OUT/'01_operator_verification'; D02=OUT/'02_roster_and_lineage'
D03=OUT/'03_runtime_outputs'; D04=OUT/'04_runtime_integrity'; D05=OUT/'05_prediction_lock'
D06=OUT/'06_paired_truth_evaluation'; D07=OUT/'07_blinded_review'; D08=OUT/'08_combined_results'; D09=OUT/'09_replay'; UPLOAD=OUT/'upload'
for p in [OUT,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Notebook 72 output:',OUT)

Notebook 72 output: /content/drive/MyDrive/MusicAnnotationProject/notebook72_runs/20261116_181813_notebook72_frozen_v8_8_operational_trial


## 1. Core helpers, access audit and atomic output controls

In [22]:
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    path=Path(path); require(path.exists() and path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),default=str).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text())
def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def imread(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(im is not None,f'Unreadable image: {path}'); return im

def semantic_npz_hash(path):
    with np.load(path,allow_pickle=False) as z: return object_hash({k:array_hash(z[k]) for k in sorted(z.files)})

def directory_inventory(root):
    rows=[]
    for p in sorted(Path(root).rglob('*')):
        if p.is_file(): rows.append({'relative_path':str(p.relative_to(root)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    return pd.DataFrame(rows)

ACCESS_ROWS=[]
LOCK_STATE='OPEN'
_original_open=builtins.open

def audited_open(file,*args,**kwargs):
    raw=str(file)
    clean_hit='/source/clean/' in raw.replace('\\','/') or raw.replace('\\','/').endswith('/source/clean')
    ACCESS_ROWS.append({'utc':datetime.now(timezone.utc).isoformat(),'lock_state':LOCK_STATE,'path':raw,'clean_path':clean_hit,'mode':str(args[0] if args else kwargs.get('mode','r'))})
    if LOCK_STATE!='CLOSED' and clean_hit: raise RuntimeError(f'TRUTH FIREWALL: clean access before lock: {raw}')
    return _original_open(file,*args,**kwargs)

@contextlib.contextmanager
def truth_firewall():
    builtins.open=audited_open
    try: yield
    finally: builtins.open=_original_open

## 2. Resolve and verify the frozen V8.8 operator bundle

The notebook fails closed rather than recreating V8.8 from historical fragments. A compatible adapter must return a page result containing `corrected`, `baseline`, `prepared_mask`, `decisions`, `probabilities`, `changed`, and `audit`.

In [24]:
def resolve_active_bundle():
    global V88_OPERATOR_PATH,V88_BUNDLE_ROOT
    if V88_BUNDLE_ROOT is None and ACTIVE_RELEASE_POINTER.exists():
        active=read_json(ACTIVE_RELEASE_POINTER)
        manifest=Path(active.get('release_manifest',''))
        if manifest.exists(): V88_BUNDLE_ROOT=manifest.parent
    require(V88_BUNDLE_ROOT is not None,'Set V88_BUNDLE_ROOT or provide an active V8.8 release pointer.')
    V88_BUNDLE_ROOT=Path(V88_BUNDLE_ROOT)
    require(V88_BUNDLE_ROOT.exists(),f'V8.8 bundle root missing: {V88_BUNDLE_ROOT}')
    if V88_OPERATOR_PATH is None:
        candidates=list(V88_BUNDLE_ROOT.rglob('v8_8_operator.py'))+list(ROOT.rglob('v8_8_operator.py'))
        candidates=list({str(p.resolve()):p.resolve() for p in candidates}.values())
        require(len(candidates)==1,f'Expected one v8_8_operator.py; found {candidates}')
        V88_OPERATOR_PATH=candidates[0]
    V88_OPERATOR_PATH=Path(V88_OPERATOR_PATH)
    require(V88_OPERATOR_PATH.exists(),f'Operator missing: {V88_OPERATOR_PATH}')

resolve_active_bundle()
spec=importlib.util.spec_from_file_location('v8_8_operator',V88_OPERATOR_PATH)
v88_module=importlib.util.module_from_spec(spec);
sys.modules['v8_8_operator'] = v88_module
spec.loader.exec_module(v88_module)
require(hasattr(v88_module,'load_operator') and hasattr(v88_module,'run_page'),'Adapter must expose load_operator and run_page.')
operator=v88_module.load_operator(V88_BUNDLE_ROOT)
operator_identity={
 'api_version':getattr(v88_module,'API_VERSION',None),
 'operator_path':str(V88_OPERATOR_PATH),'operator_sha256':sha256_file(V88_OPERATOR_PATH),
 'bundle_root':str(V88_BUNDLE_ROOT),'bundle_inventory':directory_inventory(V88_BUNDLE_ROOT).to_dict('records')
}
require(operator_identity['api_version'] in [None,EXPECTED_API_VERSION],f'Incompatible adapter API: {operator_identity["api_version"]}')
operator_identity['bundle_inventory_sha256']=object_hash(operator_identity['bundle_inventory'])
atomic_json(D01/'operator_identity.json',operator_identity)
pd.DataFrame(operator_identity['bundle_inventory']).to_csv(D01/'operator_artifact_inventory.csv',index=False)
print('Operator:',V88_OPERATOR_PATH); print('Bundle:',V88_BUNDLE_ROOT)

RuntimeError: Adapter must expose load_operator and run_page.

## 3. Frozen replay gate

In [ ]:
require(hasattr(v88_module,'replay_test'),'Adapter must expose replay_test(operator, bundle_root).')
replay=v88_module.replay_test(operator,V88_BUNDLE_ROOT)
require(isinstance(replay,dict) and replay.get('passed') is True,f'Frozen V8.8 replay failed: {replay}')
atomic_json(D09/'replay_test.json',replay)
print(json.dumps(replay,indent=2))

## 4. Build the full runtime roster and resolve authoritative review lineage

Resolution follows the hash-based HITL pattern used by the supervised predecessor notebooks. Reviewer audit records take priority. There is no fallback to unreviewed draft masks.

In [ ]:
def md5_file(path):
    h=hashlib.md5()
    with _original_open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def load_audits(folder):
    rows=[]; failures=[]
    if Path(folder).exists():
        for p in sorted(Path(folder).glob('*.json')):
            try:
                x=json.loads(p.read_text()); require(isinstance(x,dict),f'Non-object audit: {p}'); x['_audit_path']=str(p); rows.append(x)
            except Exception as e: failures.append({'audit_path':str(p),'error':str(e)})
    return rows,failures

review_audits,parse_failures=load_audits(HITL_AUDIT_DIR)
pd.DataFrame(parse_failures).to_csv(D02/'audit_parse_failures.csv',index=False)
require(not parse_failures,'Unreadable review audit records exist.')

def find_review_lineage(source):
    digest=md5_file(source)
    matches=[r for r in review_audits if str(r.get('source_md5','')).lower()==digest.lower()]
    if not matches:
        # Secondary exact source-path match for older audit schemas.
        matches=[r for r in review_audits if str(r.get('source',''))==str(source) or str(r.get('source_file',''))==str(source)]
    require(len(matches)<=1,f'{source.name}: duplicate review audits: {[x.get("_audit_path") for x in matches]}')
    if not matches: return None,'NO_REVIEW_AUDIT'
    r=matches[0]
    observed=Path(r.get('aligned_output') or source)
    mask=Path(r.get('corrected_mask_output') or '')
    if not observed.exists(): return None,'REVIEWED_OBSERVED_MISSING'
    if not mask.exists(): return None,'AUTHORITATIVE_MASK_MISSING'
    return {'observed_path':observed,'seed_mask_path':mask,'review_audit_path':Path(r['_audit_path']),'source_md5':digest},None

sources=[]
for cohort,folder in [('A_ANNOTATED_ONLY',ANNOTATED_ONLY_DIR),('B_PAIRED_AS_ANNOTATED_ONLY',ANNOTATED_DIR)]:
    for stem,path in image_map(folder).items(): sources.append({'page':stem,'cohort':cohort,'source_path':path})
# Deduplicate exact content and explicitly reject conflicting filename stems.
seen_hash={}; roster=[]; exclusions=[]
for item in sorted(sources,key=lambda x:(x['page'],x['cohort'])):
    path=item['source_path']; sha=sha256_file(path)
    if sha in seen_hash:
        exclusions.append({**item,'status':'EXCLUDED','reason':'DUPLICATE_CONTENT','duplicate_of':seen_hash[sha]}); continue
    seen_hash[sha]=item['page']
    lineage,reason=find_review_lineage(path)
    if reason: exclusions.append({**item,'status':'EXCLUDED','reason':reason}); continue
    obs=imread(lineage['observed_path']); seed=imread(lineage['seed_mask_path'])
    if obs.shape!=seed.shape:
        exclusions.append({**item,'status':'EXCLUDED','reason':'OBSERVED_MASK_SHAPE_MISMATCH'}); continue
    roster.append({**item,**lineage,'source_sha256':sha,'observed_sha256':sha256_file(lineage['observed_path']),'seed_mask_sha256':sha256_file(lineage['seed_mask_path']),'status':'ELIGIBLE'})

roster=pd.DataFrame(roster); exclusions=pd.DataFrame(exclusions)
require(len(roster)>0,'No eligible reviewed pages found.')
require(not roster.page.duplicated().any(),'Eligible roster contains duplicate page stems across cohorts. Rename or explicitly resolve them.')
if PAGE_ALLOWLIST is not None: roster=roster[roster.page.isin(set(map(str,PAGE_ALLOWLIST)))].copy()
if SMOKE_TEST: roster=roster.head(SMOKE_TEST_PAGES).copy()
roster.to_csv(D02/'runtime_roster.csv',index=False); exclusions.to_csv(D02/'exclusions.csv',index=False)
print(roster.groupby('cohort').size()); print('Excluded:',len(exclusions))

## 5. Runtime context firewall audit

In [ ]:
forbidden_tokens=['clean_path','truth','structural_safe','cosmetic_outcome','clean_sha256']
for c in roster.columns:
    require(not any(t in c.lower() for t in forbidden_tokens),f'Forbidden runtime roster field: {c}')
require('source/clean' not in json.dumps(roster.astype(str).to_dict('records')).replace('\\','/'),'Runtime roster contains clean path.')
atomic_json(D00/'configuration.json',{
 'created_utc':datetime.now(timezone.utc).isoformat(),'run_id':RUN_ID,'smoke_test':SMOKE_TEST,
 'annotated_only_dir':str(ANNOTATED_ONLY_DIR),'annotated_dir':str(ANNOTATED_DIR),
 'clean_dir_not_resolved_pre_lock':True,'operator_identity_sha256':sha256_file(D01/'operator_identity.json'),
 'random_seed':RANDOM_SEED,'promotion_authority':False,'automatic_output_authority':False
})
print('Truth path closed. Runtime roster contains no clean path or truth field.')

## 6. Full frozen inference, per-page integrity and atomic outputs

In [ ]:
@dataclass
class NormalisedResult:
    corrected: np.ndarray; baseline: np.ndarray; prepared_mask: np.ndarray
    decisions: np.ndarray; probabilities: dict; changed: np.ndarray; audit: dict

def normalise_result(result,shape):
    def get(name): return result[name] if isinstance(result,dict) else getattr(result,name)
    corrected=np.asarray(get('corrected')); baseline=np.asarray(get('baseline')); prepared=np.asarray(get('prepared_mask')).astype(bool)
    decisions=np.asarray(get('decisions')); changed=np.asarray(get('changed')).astype(bool)
    probabilities=get('probabilities') or {}; audit=get('audit') or {}
    require(corrected.shape==baseline.shape==prepared.shape==decisions.shape==changed.shape==shape,'Operator result shape mismatch.')
    require(corrected.dtype==baseline.dtype,'Corrected/baseline dtype mismatch.')
    return NormalisedResult(corrected,baseline,prepared,decisions,probabilities,changed,audit)

def write_page(page,row,result,observed,seed,elapsed):
    d=D03/page; tmp=D03/(page+'.tmp'); tmp.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(tmp/'observed.png'),observed); cv2.imwrite(str(tmp/'authoritative_seed_mask.png'),seed)
    cv2.imwrite(str(tmp/'prepared_authoritative_mask.png'),result.prepared_mask.astype(np.uint8)*255)
    cv2.imwrite(str(tmp/'fallback_corrected.png'),result.baseline); cv2.imwrite(str(tmp/'v8_8_corrected.png'),result.corrected)
    cv2.imwrite(str(tmp/'changed_pixels.png'),result.changed.astype(np.uint8)*255)
    np.savez_compressed(tmp/'decision_map.npz',decisions=result.decisions)
    np.savez_compressed(tmp/'probabilities.npz',**{str(k):np.asarray(v) for k,v in result.probabilities.items()})
    # Portable visual map of integer/string decisions.
    _,inv=np.unique(result.decisions.astype(str),return_inverse=True); dm=(inv.reshape(result.decisions.shape)*85).clip(0,255).astype(np.uint8); cv2.imwrite(str(tmp/'decision_map.png'),dm)
    outside=int(np.sum(result.corrected[~result.prepared_mask]!=observed[~result.prepared_mask]))
    nonintervention=int(np.sum(result.corrected[~result.changed]!=result.baseline[~result.changed]))
    changed_unjustified=int(np.sum((result.corrected!=result.baseline)&~result.changed))
    audit={'page':page,'cohort':row.cohort,'elapsed_seconds':elapsed,'outside_mask_drift':outside,'nonintervention_fallback_regression':nonintervention,'changed_unjustified':changed_unjustified,
           'observed_sha256':array_hash(observed),'seed_mask_sha256':array_hash(seed),'prepared_mask_sha256':array_hash(result.prepared_mask.astype(np.uint8)),
           'baseline_sha256':array_hash(result.baseline),'corrected_sha256':array_hash(result.corrected),'decisions_sha256':array_hash(result.decisions.astype(str)),
           'probabilities_semantic_sha256':object_hash({k:array_hash(np.asarray(v)) for k,v in result.probabilities.items()}),'changed_sha256':array_hash(result.changed.astype(np.uint8)),
           'operator_audit':result.audit}
    atomic_json(tmp/'outside_mask_audit.json',{'outside_mask_drift':outside,'required':0})
    atomic_json(tmp/'page_audit.json',audit)
    require(outside==0,f'{page}: outside-mask drift={outside}')
    require(nonintervention==0,f'{page}: fallback regression={nonintervention}')
    require(changed_unjustified==0,f'{page}: changed pixels not justified={changed_unjustified}')
    tmp.replace(d); return audit

page_rows=[]; failures=[]
runtime_context={'bundle_root':str(V88_BUNDLE_ROOT),'operator_identity':operator_identity,'deterministic':True,'clean_available':False}
with truth_firewall():
    for row in roster.itertuples(index=False):
        try:
            observed=imread(row.observed_path); seed=imread(row.seed_mask_path)
            t=time.time(); raw=v88_module.run_page(observed,seed,runtime_context); elapsed=time.time()-t
            result=normalise_result(raw,observed.shape); audit=write_page(str(row.page),row,result,observed,seed,elapsed)
            page_rows.append({**{k:getattr(row,k) for k in roster.columns},**{k:v for k,v in audit.items() if k!='operator_audit'},'status':'PROCESSED'})
        except Exception as e:
            failures.append({'page':row.page,'cohort':row.cohort,'status':'FAILED','reason':type(e).__name__,'details':str(e),'traceback':traceback.format_exc()})

page_integrity=pd.DataFrame(page_rows); failure_df=pd.DataFrame(failures)
page_integrity.to_csv(D04/'page_integrity.csv',index=False); failure_df.to_csv(D08/'failure_and_exception_register.csv',index=False)
pd.DataFrame(ACCESS_ROWS).to_csv(D04/'runtime_access_audit.csv',index=False)
require(len(page_integrity)+len(failure_df)==len(roster),'Runtime accounting mismatch.')
require(len(page_integrity)>0,'No pages processed successfully.')
require((page_integrity.outside_mask_drift==0).all(),'Outside-mask immutability failure.')
require((page_integrity.nonintervention_fallback_regression==0).all(),'Fallback-equivalence failure.')
require(not any(x['clean_path'] and x['lock_state']!='CLOSED' for x in ACCESS_ROWS),'Clean access occurred before lock.')
print('Processed:',len(page_integrity),'Failed:',len(failure_df))

## 7. Prediction/output lock

In [ ]:
output_rows=[]
for page in page_integrity.page.astype(str):
    d=D03/page
    output_rows.append({'page':page,'fallback_sha256':sha256_file(d/'fallback_corrected.png'),'corrected_sha256':sha256_file(d/'v8_8_corrected.png'),
                        'decisions_semantic_sha256':semantic_npz_hash(d/'decision_map.npz'),'probabilities_semantic_sha256':semantic_npz_hash(d/'probabilities.npz'),
                        'changed_sha256':sha256_file(d/'changed_pixels.png'),'page_audit_sha256':sha256_file(d/'page_audit.json')})
outputs=pd.DataFrame(output_rows).sort_values('page'); outputs.to_csv(D05/'page_output_hashes.csv',index=False)
inputs=page_integrity[['page','cohort','observed_sha256','seed_mask_sha256','prepared_mask_sha256']].sort_values('page'); inputs.to_csv(D05/'input_hashes.csv',index=False)
access=pd.read_csv(D04/'runtime_access_audit.csv'); clean_pre=int(((access.clean_path==True)&(access.lock_state!='CLOSED')).sum()) if len(access) else 0
lock_payload={'schema':'notebook72-prediction-lock-v1','runtime_roster_sha256':sha256_file(D02/'runtime_roster.csv'),'input_hashes_sha256':sha256_file(D05/'input_hashes.csv'),
              'operator_identity_sha256':sha256_file(D01/'operator_identity.json'),'page_output_hashes_sha256':sha256_file(D05/'page_output_hashes.csv'),
              'runtime_access_audit_sha256':sha256_file(D04/'runtime_access_audit.csv'),'replay_test_sha256':sha256_file(D09/'replay_test.json'),
              'processed_pages':len(page_integrity),'failed_pages':len(failure_df),'clean_reads_before_lock':clean_pre}
require(clean_pre==0,'Prediction lock refused: clean was read before lock.')
lock_id=object_hash(lock_payload); lock={'lock_id':lock_id,'state':'CLOSED','created_utc':datetime.now(timezone.utc).isoformat(),'payload':lock_payload}
atomic_json(D05/'prediction_lock.json',lock)
LOCK_STATE='CLOSED'
print('PREDICTION LOCK CLOSED:',lock_id)

## 8. Post-lock paired truth evaluation

This cell is the first place that resolves `source/clean`. It verifies the current prediction lock before loading any clean page. Metrics preserve the project distinction between structural safety and cosmetic usefulness.

In [ ]:
current_lock=read_json(D05/'prediction_lock.json')
require(current_lock.get('state')=='CLOSED' and current_lock.get('lock_id')==lock_id,'Prediction lock is not the current CLOSED lock.')
require(current_lock['payload']['page_output_hashes_sha256']==sha256_file(D05/'page_output_hashes.csv'),'Page outputs changed after lock.')

CLEAN_DIR=Path(CLEAN_DIR_LITERAL)  # first permitted resolution
clean_map=image_map(CLEAN_DIR)

def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if not ref.sum(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1); recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum(); return precision,recall,2*precision*recall/max(precision+recall,1e-12)

eval_rows=[]
for row in page_integrity.itertuples(index=False):
    page=str(row.page)
    if row.cohort!='B_PAIRED_AS_ANNOTATED_ONLY' or page not in clean_map: continue
    d=D03/page; clean=imread(clean_map[page]); observed=imread(d/'observed.png'); corrected=imread(d/'v8_8_corrected.png'); mask=imread(d/'prepared_authoritative_mask.png')>0
    if not (clean.shape==observed.shape==corrected.shape==mask.shape):
        eval_rows.append({'page':page,'scored':False,'reason':'POST_LOCK_SHAPE_MISMATCH'}); continue
    ci,oi,ri=printed_ink(clean),printed_ink(observed),printed_ink(corrected); ip,ir,if1=tolerant_scores(ci,ri,mask); ep,er,ef1=tolerant_scores(edges(clean),edges(corrected),mask)
    diff=np.abs(clean.astype(np.float32)-corrected.astype(np.float32)); removed_print=float(np.sum(ci&~ri&mask)/max(np.sum(ci&mask),1)); residual_annotation=float(np.sum(ri&~ci&mask)/max(np.sum(mask),1))
    eval_rows.append({'page':page,'cohort':row.cohort,'scored':True,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,
                      'inside_mask_mae':float(diff[mask].mean()) if mask.any() else 0.0,'whole_page_mae':float(diff.mean()),'removed_print_fraction':removed_print,'residual_annotation_fraction':residual_annotation,
                      'catastrophic_print_damage':bool(removed_print>.01 or (np.isfinite(ir) and ir<.95))})
paired=pd.DataFrame(eval_rows); paired.to_csv(D06/'paired_quantitative_results.csv',index=False)
print('Truth-scored paired pages:',int(paired.scored.sum()) if len(paired) and 'scored' in paired else 0)

## 9. Generate blinded A/B review packages for every processed page

In [ ]:
rng=random.Random(RANDOM_SEED); review_keys=[]
for row in page_integrity.itertuples(index=False):
    page=str(row.page); d=D03/page; rd=D07/page; rd.mkdir(parents=True,exist_ok=False)
    original=imread(d/'observed.png'); corrected=imread(d/'v8_8_corrected.png'); corrected_is_a=bool(rng.getrandbits(1))
    A=corrected if corrected_is_a else original; B=original if corrected_is_a else corrected
    cv2.imwrite(str(rd/'review_A.png'),A); cv2.imwrite(str(rd/'review_B.png'),B)
    key={'page':page,'cohort':row.cohort,'A':'corrected' if corrected_is_a else 'original','B':'original' if corrected_is_a else 'corrected'}
    atomic_json(rd/'review_key.json',key); review_keys.append(key)
pd.DataFrame(review_keys).to_csv(D07/'sealed_review_keys.csv',index=False)
review_template=pd.DataFrame([{'page':r['page'],'reviewer_code':'','outcome':'','annotation_removal':'','notation_damage':'','preference':'','performability':'','rejection_region':'','reason':'','reviewed_utc':''} for r in review_keys])
review_template.to_csv(D07/'performance_usability_reviews.csv',index=False)
print('Review packages:',len(review_keys))

## 10. Blinded performance-usability reviewer

Allowed outcomes: `PERFORMANCE_READY`, `PERFORMANCE_READY_WITH_MINOR_RESIDUE`, `REVIEW_REQUIRED`, `UNUSABLE_ANNOTATION_REMAINS`, `UNUSABLE_PRINT_DAMAGE`.

In [ ]:
import gradio as gr
review_df=pd.read_csv(D07/'performance_usability_reviews.csv',dtype=str).fillna('')
pages=review_df.page.tolist(); first=pages[0]
OUTCOMES=['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE','REVIEW_REQUIRED','UNUSABLE_ANNOTATION_REMAINS','UNUSABLE_PRINT_DAMAGE']

def review_pair(page): return str(D07/page/'review_A.png'),str(D07/page/'review_B.png')
def save_review(page,reviewer,outcome,removal,damage,preference,performability,region,reason):
    global review_df
    require(outcome in OUTCOMES,'Select an outcome.'); require(bool(reviewer.strip()),'Reviewer code required.')
    i=review_df.index[review_df.page==page]; require(len(i)==1,'Review page mismatch.')
    review_df.loc[i[0],:]=[page,reviewer,outcome,removal,damage,preference,performability,region,reason,datetime.now(timezone.utc).isoformat()]
    review_df.to_csv(D07/'performance_usability_reviews.csv',index=False)
    return f'Saved {page}: {outcome}'

with gr.Blocks(title='Notebook 72 Blinded Performance-Usability Review') as app:
    gr.Markdown('# Notebook 72 blinded page review\nA and B are randomised. Judge whether the corrected page is clean and trustworthy enough to perform from. Do not open diagnostic artefacts before recording the initial judgement.')
    page=gr.Dropdown(pages,value=first,label='Page')
    with gr.Row(): A=gr.Image(value=str(D07/first/'review_A.png'),label='A',height=720); B=gr.Image(value=str(D07/first/'review_B.png'),label='B',height=720)
    reviewer=gr.Textbox(label='Reviewer code'); outcome=gr.Dropdown(OUTCOMES,label='Outcome')
    removal=gr.Radio(['complete','partial','none'],label='Annotation removal'); damage=gr.Radio(['none','uncertain','visible_material_damage'],label='Notation damage')
    preference=gr.Radio(['A_better','same','B_better'],label='Preference'); performability=gr.Radio(['yes','yes_with_minor_concern','no'],label='Would you perform from the preferred page?')
    region=gr.Textbox(label='Rejection/referral region'); reason=gr.Textbox(label='Reason/notes'); save=gr.Button('Lock this initial judgement',variant='primary'); status=gr.Textbox(interactive=False)
    page.change(review_pair,[page],[A,B]); save.click(save_review,[page,reviewer,outcome,removal,damage,preference,performability,region,reason],[status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 11. Decode reviews, calculate gates and write combined decision package

In [ ]:
reviews=pd.read_csv(D07/'performance_usability_reviews.csv',dtype=str).fillna('')
require((reviews.outcome!='').all(),'Review is incomplete. Record one outcome for every processed page.')
keys=pd.read_csv(D07/'sealed_review_keys.csv',dtype=str)
reviews=reviews.merge(keys,on=['page'],validate='one_to_one')
reviews['corrected_preferred']=((reviews.preference=='A_better')&(reviews.A=='corrected'))|((reviews.preference=='B_better')&(reviews.B=='corrected'))
reviews.to_csv(D07/'performance_usability_reviews_decoded.csv',index=False)
summary=reviews.groupby(['cohort','outcome'],as_index=False).size().rename(columns={'size':'pages'}); summary.to_csv(D07/'performance_usability_summary.csv',index=False)

paired_scored=paired[paired.get('scored',False)==True] if len(paired) else pd.DataFrame()
structural_ok=bool(len(paired_scored)==0 or (~paired_scored.catastrophic_print_damage.astype(bool)).all())
no_visible_damage=not reviews.outcome.eq('UNUSABLE_PRINT_DAMAGE').any()
ready=reviews.outcome.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE'])
useful_share=float(ready.mean())
source_summary=page_integrity[['page','cohort','status']].merge(reviews[['page','outcome','corrected_preferred']],on='page',how='left')
source_summary.groupby('cohort',as_index=False).agg(pages=('page','nunique'),performance_ready=('outcome',lambda x:int(x.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE']).sum())),unusable_print_damage=('outcome',lambda x:int((x=='UNUSABLE_PRINT_DAMAGE').sum())),corrected_preferred=('corrected_preferred','sum')).to_csv(D08/'results_by_source_group.csv',index=False)
source_summary.to_csv(D08/'results_by_page.csv',index=False)

gates={
 'G1_frozen_identity':replay.get('passed') is True,
 'G2_roster_completeness':len(page_integrity)+len(failure_df)==len(roster),
 'G3_truth_firewall':clean_pre==0,
 'G4_outside_mask_immutability':bool((page_integrity.outside_mask_drift==0).all()),
 'G5_fallback_equivalence':bool((page_integrity.nonintervention_fallback_regression==0).all()),
 'G6_replay':replay.get('passed') is True,
 'G7_structural_safety':bool(structural_ok and no_visible_damage),
 'G8_cosmetic_usefulness':bool(useful_share>0),
 'G9_source_stability':'REVIEW_REQUIRED_HUMAN_INTERPRETATION',
 'G10_auditability':bool(outputs.page.nunique()==len(page_integrity) and (page_integrity.changed_unjustified==0).all())
}
required_bool=['G1_frozen_identity','G2_roster_completeness','G3_truth_firewall','G4_outside_mask_immutability','G5_fallback_equivalence','G6_replay','G7_structural_safety','G8_cosmetic_usefulness','G10_auditability']
first_failed=next((k for k in required_bool if gates[k] is not True),None)
verdict='PASS_SUPERVISED_OPERATIONAL_TRIAL_EVIDENCE_READY_FOR_DECISION' if first_failed is None else 'FAIL_NOTEBOOK72_OPERATIONAL_TRIAL'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'lock_id':lock_id,'eligible_pages':len(roster),'processed_pages':len(page_integrity),'failed_pages':len(failure_df),'excluded_pages':len(exclusions),'paired_truth_scored':len(paired_scored),'performance_ready_share':useful_share,'promotion_authority':False,'automatic_output_authority':False,'mask_authority_automation':False}
atomic_json(OUT/'notebook72_verdict.json',record)
atomic_json(OUT/'experiment_manifest.json',{'run_id':RUN_ID,'operator_identity':operator_identity,'prediction_lock':lock,'replay':replay,'record':record})
lines=['NOTEBOOK 72 / FROZEN V8.8 ANNOTATED-ONLY SUPERVISED OPERATIONAL TRIAL','',f'Verdict: {verdict}',f'Prediction lock: {lock_id}',f'Eligible / processed / failed / excluded: {len(roster)} / {len(page_integrity)} / {len(failure_df)} / {len(exclusions)}',f'Paired truth-scored pages: {len(paired_scored)}',f'Performance-ready share: {useful_share:.3%}','','GATES']+[f'- {k}: {v}' for k,v in gates.items()]+['','No retraining, recalibration, threshold movement, automatic mask authority or production release was performed.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines))

## 12. Standard Final Upload Package

In [ ]:
for sub in ['tables','review','visuals','audit']: (UPLOAD/sub).mkdir(parents=True,exist_ok=True)
curated=[OUT/'executive_summary.txt',OUT/'notebook72_verdict.json',OUT/'experiment_manifest.json',D05/'prediction_lock.json',D02/'runtime_roster.csv',D02/'exclusions.csv',D04/'page_integrity.csv',D04/'runtime_access_audit.csv',D05/'page_output_hashes.csv',D06/'paired_quantitative_results.csv',D07/'performance_usability_reviews_decoded.csv',D07/'performance_usability_summary.csv',D08/'results_by_page.csv',D08/'results_by_source_group.csv',D08/'failure_and_exception_register.csv',D09/'replay_test.json',D01/'operator_artifact_inventory.csv']
rows=[]
for src in curated:
    require(src.exists(),f'Missing required upload artefact: {src}')
    group='review' if 'performance_usability' in src.name else ('audit' if src.suffix=='.json' or 'audit' in src.name or 'integrity' in src.name or 'hash' in src.name else 'tables')
    dst=UPLOAD/group/src.name; shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
# Include low-resolution A/B review images, but keep full runtime pages in the run directory.
for page in page_integrity.page.astype(str):
    for name in ['review_A.png','review_B.png']:
        src=D07/page/name; dst=UPLOAD/'review'/f'{page}_{name}'; shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
manifest=pd.DataFrame(rows).sort_values('relative_path'); manifest.to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(manifest.relative_path))
(UPLOAD/'README.txt').write_text('Notebook 72 review package. Full-resolution runtime outputs remain under 03_runtime_outputs and are indexed by page_output_hashes.csv. Clean truth was unavailable to runtime code before the CLOSED prediction lock. This package has no production-release authority.\n')
zip_path=UPLOAD/'notebook72_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print('Upload bundle:',zip_path,'SHA-256:',sha256_file(zip_path))

## Outputs to inspect first
- `executive_summary.txt`
- `notebook72_verdict.json`
- `05_prediction_lock/prediction_lock.json`
- `04_runtime_integrity/page_integrity.csv`
- `06_paired_truth_evaluation/paired_quantitative_results.csv`
- `07_blinded_review/performance_usability_summary.csv`
- `08_combined_results/failure_and_exception_register.csv`
- `upload/notebook72_upload_bundle.zip`

If operator resolution, replay, truth firewall, immutability or fallback equivalence fails, do not suppress the error. Fix the frozen artefact or lineage issue and create a new run.